# 🕉️ Sanskrit MNIST — Exploration & Training Notebook

This notebook walks through the full pipeline:
1. Dataset exploration
2. Model architecture overview
3. Training
4. Evaluation
5. Per-class analysis

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

from src.dataset import get_dataloaders, load_label_map
from src.model import get_model, count_parameters
from src.utils import get_device, set_seed

set_seed(42)
device = get_device()
print(f'PyTorch version: {torch.__version__}')

## 1 — Dataset Overview

In [ ]:
label_map = load_label_map('../data/raw/label_map.csv')

print(f'Total classes : {len(label_map)}')
print()

categories = {}
for k, v in label_map.items():
    cat = v['category']
    categories.setdefault(cat, []).append(v['char'])

for cat, chars in categories.items():
    print(f'{cat.capitalize():12s} ({len(chars):2d}): {" ".join(chars)}')

In [ ]:
# Load dataloaders
train_loader, val_loader, test_loader, label_map = get_dataloaders(
    data_dir='../data/raw',
    batch_size=64,
    seed=42,
)

# Inspect one batch
images, labels = next(iter(train_loader))
print(f'Batch shape  : {images.shape}')  # (64, 1, 32, 32)
print(f'Labels shape : {labels.shape}')
print(f'Pixel range  : [{images.min():.2f}, {images.max():.2f}]')  # [-1, 1]

In [ ]:
# Visualise a sample batch
fig, axes = plt.subplots(4, 8, figsize=(14, 7))
fig.suptitle('Sample training images (de-normalised)', fontsize=12, fontweight='bold')

for idx, ax in enumerate(axes.flat):
    if idx >= 32:
        break
    img = images[idx].squeeze().numpy()
    img = (img * 0.5) + 0.5   # de-normalise
    lbl = label_map[labels[idx].item()]
    ax.imshow(img, cmap='gray')
    ax.set_title(f"{lbl['char']}\n{lbl['roman']}", fontsize=7)
    ax.axis('off')

plt.tight_layout()
plt.show()

## 2 — Model Architecture

In [ ]:
model = get_model(num_classes=62)
print(model)
print(f'\nTrainable parameters: {count_parameters(model):,}')

# Dry run
dummy = torch.randn(1, 1, 32, 32)
out = model(dummy)
print(f'Output shape: {out.shape}')  # (1, 62)

## 3 — Training

Run `python train.py` from the terminal for the full training loop.  
Below is a minimal in-notebook training run (5 epochs) for demonstration.

In [ ]:
import torch.nn as nn
from src.utils import train_one_epoch, evaluate, MetricsTracker

model = get_model(num_classes=62).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()
tracker = MetricsTracker()

DEMO_EPOCHS = 5

for epoch in range(1, DEMO_EPOCHS + 1):
    tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, criterion, device)
    v_loss,  v_acc  = evaluate(model, val_loader, criterion, device)
    tracker.update(tr_loss, tr_acc, v_loss, v_acc, epoch_time=0)
    print(f'Epoch {epoch}/{DEMO_EPOCHS}  '
          f'train_acc={tr_acc*100:.1f}%  val_acc={v_acc*100:.1f}%')

In [ ]:
# Plot training curves inline
h = tracker.history
epochs = range(1, len(h['train_loss']) + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.plot(epochs, h['train_loss'], label='Train'); ax1.plot(epochs, h['val_loss'], label='Val', linestyle='--')
ax1.set_title('Loss'); ax1.legend(); ax1.grid(alpha=0.3)
ax2.plot(epochs, [a*100 for a in h['train_acc']], label='Train'); ax2.plot(epochs, [a*100 for a in h['val_acc']], label='Val', linestyle='--')
ax2.set_title('Accuracy (%)'); ax2.legend(); ax2.grid(alpha=0.3)
plt.suptitle('Training Curves (demo — 5 epochs)', fontweight='bold')
plt.tight_layout(); plt.show()

## 4 — Evaluation on Test Set

In [ ]:
# Load the best model trained via train.py
from src.utils import load_checkpoint

best_model = get_model(num_classes=62).to(device)
try:
    ckpt = load_checkpoint('../data/processed/best_model.pt', best_model, device)
    test_loss, test_acc = evaluate(best_model, test_loader, criterion, device)
    print(f'Test accuracy : {test_acc*100:.2f}%')
    print(f'Test loss     : {test_loss:.4f}')
    print(f'Best val acc  : {ckpt["val_acc"]*100:.2f}%  (epoch {ckpt["epoch"]})')
except FileNotFoundError:
    print('No checkpoint found — run `python train.py` first.')

## 5 — Per-class Accuracy

In [ ]:
import torch.nn.functional as F

@torch.no_grad()
def per_class_accuracy(model, loader, num_classes, device):
    model.eval()
    correct = torch.zeros(num_classes)
    total   = torch.zeros(num_classes)
    for images, labels in loader:
        preds = model(images.to(device)).argmax(dim=1).cpu()
        for c in range(num_classes):
            mask = labels == c
            total[c]   += mask.sum()
            correct[c] += (preds[mask] == c).sum()
    return (correct / total.clamp(min=1)).numpy()

try:
    accs = per_class_accuracy(best_model, test_loader, 62, device)

    fig, ax = plt.subplots(figsize=(20, 4))
    colors = ['#4C72B0' if label_map[i]['category']=='vowel'
              else '#DD8452' if label_map[i]['category']=='consonant'
              else '#59A14F' for i in range(62)]
    ax.bar([label_map[i]['roman'] for i in range(62)], accs * 100, color=colors)
    ax.axhline(accs.mean() * 100, color='red', linestyle='--', label=f'Mean {accs.mean()*100:.1f}%')
    ax.set_ylabel('Accuracy (%)')
    ax.set_title('Per-class Test Accuracy', fontweight='bold')
    ax.set_xticks(range(62))
    ax.set_xticklabels([label_map[i]['roman'] for i in range(62)], rotation=90, fontsize=7)
    ax.legend(); ax.grid(axis='y', alpha=0.3)
    plt.tight_layout(); plt.show()

    # Worst 5 classes
    worst_idx = accs.argsort()[:5]
    print('5 hardest classes:')
    for i in worst_idx:
        lbl = label_map[int(i)]
        print(f"  {lbl['char']:3s} ({lbl['roman']:8s}) — {accs[i]*100:.1f}%")
except:
    print('Train the model first.')